# LeetCode 1346 — Check If N and Its Double Exist bằng Python

**Link đề:** https://leetcode.com/problems/check-if-n-and-its-double-exist/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Sorting, Binary Search, Two Pointers

```python
class Solution:
    def checkIfExist(self, arr: list[int]) -> bool:
        ...
```

## Mục tiêu học
1. Nhận ra đây là "họ hàng" của **Two Sum**: thay `a + b = target` bằng `a = 2·b`.
2. Dùng **`set`** ghi nhớ các số đã gặp → kiểm tra `2x` và `x/2` trong `O(1)` trung bình.
3. Xử lý **trường hợp đặc biệt số 0** (`0 = 2·0`) — bẫy quen thuộc của bài.
4. So sánh 5 cách: brute force, set một lượt, `Counter`, sắp xếp + binary search, sắp xếp + hai con trỏ.
5. Trace, chứng minh, Time/Space, lỗi hay gặp, bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho mảng số nguyên `arr`, kiểm tra xem có tồn tại hai chỉ số `i`, `j` sao cho:
- `i != j`,
- `0 <= i, j < arr.length`,
- `arr[i] == 2 * arr[j]`.

| `arr` | Output | Giải thích |
|---|---|---|
| `[10,2,5,3]` | `true` | `i = 0, j = 2`: `10 == 2 · 5` |
| `[3,1,7,11]` | `false` | không cặp nào |

**Constraints:** `2 <= arr.length <= 500`, `-10^3 <= arr[i] <= 10^3`.

Câu hỏi trước khi code:
- `[0, 0]` trả về gì? Còn `[0, 1]`? (gợi ý: `0 == 2 · 0` nhưng phải **hai chỉ số khác nhau**)
- Số âm: `[-2, -4]`? `-4 == 2 · (-2)` ✔.
- `[7, 3]`: `7 / 2 = 3.5` — có phải là "nửa" của 7 không?

## 2. Nền tảng Python

### 2.1. `set` — tập hợp, tra cứu nhanh

- Tạo: `seen = set()` (chú ý: `{}` là **dict rỗng**, không phải set rỗng).
- Thêm: `seen.add(x)`.
- Kiểm tra: `x in seen` — trung bình `O(1)` (so với `x in list` là `O(n)`).

In [1]:
seen = set()
print(type(seen), type({}))
seen.add(5)
seen.add(10)
seen.add(5)            # thêm trùng không có tác dụng
print(seen, "| 10 in seen?", 10 in seen, "| 7 in seen?", 7 in seen)

<class 'set'> <class 'dict'>
{10, 5} | 10 in seen? True | 7 in seen? False


### 2.2. "Nửa của x" chỉ tồn tại khi `x` chẵn

Ta cần số nguyên `y` với `x == 2·y`. Điều này chỉ có khi `x % 2 == 0`, và khi đó `y = x // 2`.

Chú ý số âm: trong Python `-7 % 2 == 1` và `-6 % 2 == 0` — kiểm tra chẵn vẫn đúng. `-6 // 2 == -3` ✔.

In [2]:
for x in [10, 7, -6, -7, 0]:
    even = x % 2 == 0
    print(f"x={x:>3}: chẵn? {even!s:5} | nửa = {x // 2 if even else 'không có'} | 2x = {2 * x}")
print("cẩn thận: 7 // 2 =", 7 // 2, "nhưng 2 * 3 =", 2 * 3, "!= 7")

x= 10: chẵn? True  | nửa = 5 | 2x = 20
x=  7: chẵn? False | nửa = không có | 2x = 14
x= -6: chẵn? True  | nửa = -3 | 2x = -12
x= -7: chẵn? False | nửa = không có | 2x = -14
x=  0: chẵn? True  | nửa = 0 | 2x = 0
cẩn thận: 7 // 2 = 3 nhưng 2 * 3 = 6 != 7


### 2.3. `collections.Counter` — đếm số lần xuất hiện

`Counter([0, 0, 5])` → `Counter({0: 2, 5: 1})`. Truy cập `c[key]` trả về số lần, key không có trả `0` (không lỗi).

In [3]:
from collections import Counter
c = Counter([0, 0, 5, 10])
print(c, "| c[0] =", c[0], "| c[99] =", c[99])

Counter({0: 2, 5: 1, 10: 1}) | c[0] = 2 | c[99] = 0


## 3. Cách trực giác / baseline — thử mọi cặp `(i, j)`

Hai vòng lặp, bỏ qua `i == j`, kiểm tra `arr[i] == 2 * arr[j]`.

In [4]:
def check_brute(arr: list[int]) -> bool:
    n = len(arr)
    for i in range(n):
        for j in range(n):
            if i != j and arr[i] == 2 * arr[j]:
                return True
    return False

for arr in [[10, 2, 5, 3], [3, 1, 7, 11], [0, 0], [0, 1], [-2, -4], [7, 3]]:
    print(arr, "->", check_brute(arr))

[10, 2, 5, 3] -> True
[3, 1, 7, 11] -> False
[0, 0] -> True
[0, 1] -> False
[-2, -4] -> True
[7, 3] -> False


### 3.1. Nút thắt

`n²` cặp: với `n = 500` là 250 000 phép so sánh — vẫn qua được LeetCode, nhưng nếu `n = 10^5` thì `10^10` — quá chậm.

Công việc lặp lại: với mỗi `arr[j]`, ta **quét lại cả mảng** để tìm `2·arr[j]`. Câu hỏi giống Two Sum: *có cách nào trả lời "giá trị `v` có trong mảng không?" trong `O(1)`?* → **set**.

## 4. Insight — duyệt một lượt, nhớ các số đã gặp

Đi từ trái sang phải. Tại `x`, các số **đứng trước** nằm trong `seen`. Cặp hợp lệ với `x` là:
- `x` là **gấp đôi** của một số trước: cần `x / 2` trong `seen` (chỉ khi `x` chẵn).
- `x` là **một nửa** của một số trước: cần `2 * x` trong `seen`.

Kiểm tra **trước** rồi mới `add(x)` → không bao giờ ghép `x` với chính nó (đảm bảo `i != j`). Vì mọi cặp `(i, j)` đều có một phần tử đứng sau, cặp sẽ được phát hiện khi duyệt tới phần tử đứng sau đó.

In [5]:
arr = [10, 2, 5, 3]
seen = set()
for x in arr:
    need_double, need_half = 2 * x, (x // 2 if x % 2 == 0 else None)
    print(f"x={x}: seen={seen}, cần {need_double} (2x) hoặc {need_half} (x/2)")
    seen.add(x)

x=10: seen=set(), cần 20 (2x) hoặc 5 (x/2)
x=2: seen={10}, cần 4 (2x) hoặc 1 (x/2)
x=5: seen={10, 2}, cần 10 (2x) hoặc None (x/2)
x=3: seen={10, 2, 5}, cần 6 (2x) hoặc None (x/2)


## 5. Lời giải chính — `set` một lượt, giảng từng đoạn

**Đoạn 1:** `seen = set()`.

**Đoạn 2:** với mỗi `x` trong `arr`:
- nếu `2 * x in seen` → `True`;
- nếu `x % 2 == 0 and x // 2 in seen` → `True`.

**Đoạn 3:** `seen.add(x)`.

**Đoạn 4:** hết mảng → `False`.

Số 0 tự động đúng: `[0, 0]` — gặp `0` đầu, `seen` rỗng, thêm `0`; gặp `0` thứ hai, `2 * 0 = 0 in seen` → `True`. Còn `[0, 1]` chỉ có một số 0 → không ghép với chính nó → `False`.

In [6]:
def check_set(arr: list[int]) -> bool:
    seen = set()
    for x in arr:
        if 2 * x in seen:
            return True
        if x % 2 == 0 and x // 2 in seen:
            return True
        seen.add(x)
    return False

for arr in [[10, 2, 5, 3], [3, 1, 7, 11], [0, 0], [0, 1], [-2, -4], [7, 3]]:
    print(arr, "->", check_set(arr))

[10, 2, 5, 3] -> True
[3, 1, 7, 11] -> False
[0, 0] -> True
[0, 1] -> False
[-2, -4] -> True
[7, 3] -> False


## 6. TRACE từng bước

Trace `arr = [3, 1, 7, 6]` (cặp `6 = 2·3`, phát hiện khi gặp `6`):

| x | seen trước | 2x in seen? | x chẵn & x/2 in seen? | hành động |
|---|---|---|---|---|
| 3 | `{}` | 6 ✘ | lẻ | thêm 3 |
| 1 | `{3}` | 2 ✘ | lẻ | thêm 1 |
| 7 | `{1,3}` | 14 ✘ | lẻ | thêm 7 |
| 6 | `{1,3,7}` | 12 ✘ | 3 ✔ | **True** |

Trace `arr = [4, 2]`: gặp `4` → thêm; gặp `2` → `2·2 = 4 in seen` ✔ → `True` (lần này phát hiện qua nhánh "gấp đôi").

In [7]:
def trace_check(arr: list[int]) -> bool:
    seen = set()
    print("arr =", arr)
    for idx, x in enumerate(arr):
        dbl = 2 * x in seen
        half = x % 2 == 0 and x // 2 in seen
        print(f"  i={idx}, x={x:>3}, seen={sorted(seen)}, 2x={2 * x} có? {dbl}, "
              f"x/2={x // 2 if x % 2 == 0 else '-'} có? {half}")
        if dbl or half:
            print("  -> tìm thấy cặp, trả True")
            return True
        seen.add(x)
    print("  -> duyệt hết, trả False")
    return False

trace_check([3, 1, 7, 6])
print()
trace_check([4, 2])
print()
trace_check([0, 1, 3])

arr = [3, 1, 7, 6]
  i=0, x=  3, seen=[], 2x=6 có? False, x/2=- có? False
  i=1, x=  1, seen=[3], 2x=2 có? False, x/2=- có? False
  i=2, x=  7, seen=[1, 3], 2x=14 có? False, x/2=- có? False
  i=3, x=  6, seen=[1, 3, 7], 2x=12 có? False, x/2=3 có? True
  -> tìm thấy cặp, trả True

arr = [4, 2]
  i=0, x=  4, seen=[], 2x=8 có? False, x/2=2 có? False
  i=1, x=  2, seen=[4], 2x=4 có? True, x/2=1 có? False
  -> tìm thấy cặp, trả True

arr = [0, 1, 3]
  i=0, x=  0, seen=[], 2x=0 có? False, x/2=0 có? False
  i=1, x=  1, seen=[0], 2x=2 có? False, x/2=- có? False
  i=2, x=  3, seen=[0, 1], 2x=6 có? False, x/2=- có? False
  -> duyệt hết, trả False


False

## 7. Code nộp LeetCode

In [8]:
class Solution:
    def checkIfExist(self, arr: list[int]) -> bool:
        seen = set()
        for x in arr:
            if 2 * x in seen or (x % 2 == 0 and x // 2 in seen):
                return True
            seen.add(x)
        return False

sol = Solution()
print(sol.checkIfExist([10, 2, 5, 3]), sol.checkIfExist([3, 1, 7, 11]))

True False


## 8. Các lời giải khác

### Cách 3 — `Counter`: dựng bảng đếm trước, rồi kiểm tra

Với mỗi `x`:
- nếu `x == 0`: cần **ít nhất hai** số 0 (`c[0] >= 2`);
- nếu `x != 0`: cần `c[2 * x] >= 1` (vì `2x != x`, chắc chắn là chỉ số khác).

In [9]:
def check_counter(arr: list[int]) -> bool:
    c = Counter(arr)
    for x in c:                      # duyệt các giá trị khác nhau
        if x == 0:
            if c[0] >= 2:
                return True
        elif c[2 * x] > 0:
            return True
    return False

for arr in [[10, 2, 5, 3], [3, 1, 7, 11], [0, 0], [0, 1], [-2, -4]]:
    print(arr, "->", check_counter(arr))

[10, 2, 5, 3] -> True
[3, 1, 7, 11] -> False
[0, 0] -> True
[0, 1] -> False
[-2, -4] -> True


### Cách 4 — sắp xếp + binary search

Sắp xếp mảng, với mỗi vị trí `j`, tìm `2 * arr[j]` bằng binary search (`bisect_left`). Phải đảm bảo vị trí tìm được **khác `j`** — quan trọng với số `0`.

In [10]:
from bisect import bisect_left

def check_sort_bs(arr: list[int]) -> bool:
    a = sorted(arr)
    n = len(a)
    for j in range(n):
        target = 2 * a[j]
        i = bisect_left(a, target)
        # bisect_left trả vị trí đầu tiên có giá trị >= target
        if i < n and a[i] == target:
            if i != j:
                return True
            if i + 1 < n and a[i + 1] == target:     # target == a[j] (chỉ xảy ra khi a[j] = 0)
                return True
    return False

for arr in [[10, 2, 5, 3], [3, 1, 7, 11], [0, 0], [0, 1], [-2, -4]]:
    print(arr, "->", check_sort_bs(arr))

[10, 2, 5, 3] -> True
[3, 1, 7, 11] -> False
[0, 0] -> True
[0, 1] -> False
[-2, -4] -> True


### Cách 5 — sắp xếp + hai con trỏ

Sau khi sắp xếp, tách thành hai nửa: **số không âm** (tăng dần) và **số âm**. Với số không âm, nếu `a ≤ b` thì `2a ≤ 2b`, nên con trỏ `j` tìm `2·a[i]` chỉ đi tới, không lùi. Với số âm, đảo lại (dùng trị tuyệt đối) để có cùng tính chất.

Ý tưởng tổng quát: trên dãy **trị tuyệt đối đã sắp xếp**, `2·|x|` cũng tăng theo `|x|` → hai con trỏ. Ta làm riêng cho hai dấu vì `2·x` cùng dấu với `x`.

In [11]:
def has_double_sorted_nonneg(vals: list[int]) -> bool:
    # vals: các số >= 0 đã sắp tăng dần
    j = 0
    n = len(vals)
    for i in range(n):
        while j < n and vals[j] < 2 * vals[i]:
            j += 1
        if j < n and vals[j] == 2 * vals[i] and j != i:
            return True
        # trường hợp vals[i] = 0: cần một số 0 khác ở chỉ số khác i
        if vals[i] == 0 and i + 1 < n and vals[i + 1] == 0:
            return True
    return False

def check_two_pointers(arr: list[int]) -> bool:
    nonneg = sorted(x for x in arr if x >= 0)
    neg_abs = sorted(-x for x in arr if x < 0)
    return has_double_sorted_nonneg(nonneg) or has_double_sorted_nonneg(neg_abs)

for arr in [[10, 2, 5, 3], [3, 1, 7, 11], [0, 0], [0, 1], [-2, -4], [-3, 6]]:
    print(arr, "->", check_two_pointers(arr))

[10, 2, 5, 3] -> True
[3, 1, 7, 11] -> False
[0, 0] -> True
[0, 1] -> False
[-2, -4] -> True
[-3, 6] -> False


### Code nộp LeetCode cho các cách khác

In [12]:
class SolutionBrute:
    def checkIfExist(self, arr: list[int]) -> bool:
        n = len(arr)
        return any(i != j and arr[i] == 2 * arr[j] for i in range(n) for j in range(n))

class SolutionCounter:
    def checkIfExist(self, arr: list[int]) -> bool:
        c = Counter(arr)
        return c[0] >= 2 or any(x != 0 and c[2 * x] for x in c)

class SolutionSortBS:
    def checkIfExist(self, arr: list[int]) -> bool:
        a = sorted(arr)
        n = len(a)
        for j in range(n):
            t = 2 * a[j]
            i = bisect_left(a, t)
            if i < n and a[i] == t and (i != j or (i + 1 < n and a[i + 1] == t)):
                return True
        return False

for cls in [SolutionBrute, SolutionCounter, SolutionSortBS]:
    print(cls.__name__, cls().checkIfExist([10, 2, 5, 3]), cls().checkIfExist([3, 1, 7, 11]),
          cls().checkIfExist([0, 0]), cls().checkIfExist([0, 1]))

SolutionBrute True False True False
SolutionCounter True False True False
SolutionSortBS True False True False


`any(...)` trả `True` ngay khi gặp phần tử đúng đầu tiên (dừng sớm), ngược lại `False`.

### Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | mọi cặp `i != j` | `O(n²)` | `O(1)` | dễ nhất, đủ với `n <= 500` |
| 2. Set một lượt | kiểm `2x`, `x/2` trước khi thêm | `O(n)` TB | `O(n)` | lời giải chính |
| 3. Counter | đếm, xử lý riêng số 0 | `O(n)` TB | `O(n)` | rõ ràng về số 0 |
| 4. Sort + binary search | tìm `2·a[j]` | `O(n log n)` | `O(n)` | không cần hash |
| 5. Sort + hai con trỏ | tách dấu, con trỏ chỉ tiến | `O(n log n)` | `O(n)` | luyện two pointers |

TB = trung bình (tra cứu hash trung bình `O(1)`, không tuyệt đối trong mọi tình huống).

## 9. Vì sao lời giải chính đúng?

**Invariant:** trước khi xử lý phần tử thứ `t`, `seen` chứa đúng tập giá trị `{arr[0], ..., arr[t-1]}`, và chưa có cặp hợp lệ nào nằm hoàn toàn trong đoạn `arr[0..t-1]` (nếu có ta đã trả `True`).

- **Không bỏ sót:** giả sử có cặp `(i, j)`, `i != j`, `arr[i] = 2·arr[j]`. Gọi `t = max(i, j)`, phần tử kia ở chỉ số `< t` nên đã trong `seen` khi xử lý `t`. Nếu `t = i` thì `x = arr[i]` chẵn và `x // 2 = arr[j] ∈ seen`; nếu `t = j` thì `2·x = arr[i] ∈ seen`. Một trong hai nhánh bắt được.
- **Không trả nhầm:** khi trả `True`, giá trị tìm thấy trong `seen` đến từ một chỉ số **trước** `t`, nên khác `t` ⇒ `i != j` ✔. Nhánh `x // 2` chỉ dùng khi `x` chẵn nên `x == 2·(x // 2)` thật sự.

Số 0 không cần xử lý riêng: hai số 0 ở hai chỉ số khác nhau được bắt nhờ `seen`; một số 0 đơn lẻ không bao giờ tự ghép vì ta kiểm tra **trước** khi `add`. ∎

### 9.1. Độ phức tạp

- **Time:** `O(n)` trung bình — mỗi phần tử 2 lần tra set + 1 lần thêm.
- **Space:** `O(n)` cho `seen`.

## 10. Bộ test

Mẫu đề, số 0 (một / hai), số âm, số lẻ không có "nửa" nguyên, phần tử trùng khác 0, cặp ở hai đầu mảng, và **đối chiếu** mọi cách với brute force trên dữ liệu ngẫu nhiên (miền giá trị nhỏ để có nhiều cặp và nhiều số 0).

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS -> {actual}")

check("ví dụ 1", sol.checkIfExist([10, 2, 5, 3]), True)
check("ví dụ 2", sol.checkIfExist([3, 1, 7, 11]), False)
check("hai số 0", sol.checkIfExist([0, 0]), True)
check("một số 0", sol.checkIfExist([0, 1]), False)
check("số âm", sol.checkIfExist([-2, -4]), True)
check("lẻ, không có nửa", sol.checkIfExist([7, 3]), False)
check("trùng khác 0", sol.checkIfExist([5, 5]), False)
check("cặp ở hai đầu", sol.checkIfExist([1, 9, 9, 9, 2]), True)
check("khác dấu", sol.checkIfExist([-3, 6]), False)
check("biên ±1000", sol.checkIfExist([-1000, 1000, 500]), True)

ví dụ 1                PASS -> True
ví dụ 2                PASS -> False
hai số 0               PASS -> True
một số 0               PASS -> False
số âm                  PASS -> True
lẻ, không có nửa       PASS -> False
trùng khác 0           PASS -> False
cặp ở hai đầu          PASS -> True
khác dấu               PASS -> False
biên ±1000             PASS -> True


In [14]:
import random
rng = random.Random(1346)
methods = [check_set, check_counter, check_sort_bs, check_two_pointers, sol.checkIfExist,
           SolutionBrute().checkIfExist, SolutionCounter().checkIfExist, SolutionSortBS().checkIfExist]
stats = Counter()
for _ in range(3000):
    arr = [rng.randint(-10, 10) for _ in range(rng.randint(2, 8))]
    expected = check_brute(arr)
    stats[expected] += 1
    for f in methods:
        assert f(arr) == expected, (f, arr)
print("3000 mảng ngẫu nhiên: mọi cách khớp brute force | phân bố đáp án:", dict(stats))

3000 mảng ngẫu nhiên: mọi cách khớp brute force | phân bố đáp án: {False: 1909, True: 1091}


## 11. Lỗi người mới hay gặp

### Lỗi 1: Bỏ hết vào `set` trước rồi mới kiểm → số 0 tự ghép với chính nó

In [15]:
def wrong_set_first(arr):
    s = set(arr)
    for x in arr:
        if 2 * x in s:            # với x = 0: 0 in s luôn đúng!
            return True
    return False

print("[0, 1] sai:", wrong_set_first([0, 1]), "| đúng:", check_set([0, 1]))

[0, 1] sai: True | đúng: False


**Sửa:** kiểm tra trước khi thêm (cách 2), hoặc đếm số 0 riêng (cách 3).

### Lỗi 2: Chỉ kiểm `2 * x`, quên `x / 2`

Khi duyệt một lượt, nếu số **lớn** (gấp đôi) xuất hiện **trước** số nhỏ thì chỉ nhánh `2 * x` bắt được; ngược lại thì chỉ nhánh `x / 2` bắt được. Thiếu một nhánh sẽ bỏ sót.

In [16]:
def wrong_only_double(arr):
    seen = set()
    for x in arr:
        if 2 * x in seen:
            return True
        seen.add(x)
    return False

print("[2, 4] sai:", wrong_only_double([2, 4]), "| đúng:", check_set([2, 4]))
print("[4, 2]    :", wrong_only_double([4, 2]), "(thứ tự này thì may mắn đúng)")

[2, 4] sai: False | đúng: True
[4, 2]    : True (thứ tự này thì may mắn đúng)


### Lỗi 3: Dùng `x // 2` mà quên kiểm tra `x` chẵn

`7 // 2 = 3` (làm tròn xuống), nên nếu `3` đã có trong `seen` thì ta báo nhầm "`7` gấp đôi `3`".

(Ghi chú: nếu dùng chia thực `x / 2` thì `7 / 2 = 3.5` không khớp số nguyên nào nên không sai; nhưng code khó đọc hơn vì trộn `float` với `int` — ví dụ `3.0 in {3}` là `True` do `3.0 == 3`. Cách rõ ràng nhất: kiểm `x % 2 == 0` rồi dùng `x // 2`.)

In [17]:
def wrong_floor_half(arr):
    seen = set()
    for x in arr:
        if 2 * x in seen or x // 2 in seen:     # quên kiểm x chẵn
            return True
        seen.add(x)
    return False

print("[3, 7] sai:", wrong_floor_half([3, 7]), "| đúng:", check_set([3, 7]))
print("3.0 in {3}:", 3.0 in {3})

[3, 7] sai: True | đúng: False
3.0 in {3}: True


### Lỗi 4: Brute force quên `i != j`

`arr[i] == 2 * arr[i]` đúng khi `arr[i] = 0` → `[0, 1]` trả nhầm `True`.

## 12. Mẹo nhận dạng pattern

- "Có tồn tại hai phần tử thoả quan hệ `f(a) = b`" ⇒ **set/dict các phần tử đã gặp**, kiểm tra trước khi thêm (khuôn Two Sum).
- Quan hệ không đối xứng (gấp đôi ≠ một nửa) ⇒ kiểm **cả hai chiều** khi duyệt một lượt.
- Có `i != j` ⇒ luôn nghĩ tới **điểm bất động** `f(x) = x` (ở đây `x = 0`).
- Không dùng hash được ⇒ sắp xếp + binary search / hai con trỏ.

## 13. Biến thể

1. **Bội `k` bất kỳ**: `arr[i] == k · arr[j]` — kiểm `k·x` và `x / k` (khi `x % k == 0`); `k = 1` thành bài "có phần tử trùng" (LeetCode 217).
2. **Hiệu bằng `d`**: `arr[i] − arr[j] == d` — kiểm `x + d`, `x − d` (LeetCode 532 K-diff Pairs).
3. **LeetCode 954 — Array of Doubled Pairs** (Medium): ghép **toàn bộ** mảng thành các cặp `(x, 2x)` — tham lam theo trị tuyệt đối + `Counter`.

In [18]:
def check_multiple(arr: list[int], k: int) -> bool:
    seen = set()
    for x in arr:
        if k * x in seen or (x % k == 0 and x // k in seen):
            return True
        seen.add(x)
    return False

print("k=3, [2, 7, 6]:", check_multiple([2, 7, 6], 3))
print("k=1, [1, 2, 1] (có trùng?):", check_multiple([1, 2, 1], 1))
print("k=1, [1, 2, 3]:", check_multiple([1, 2, 3], 1))

k=3, [2, 7, 6]: True
k=1, [1, 2, 1] (có trùng?): True
k=1, [1, 2, 3]: False


## 14. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** `[-10, 12, -20, -8, 15]` → True/False? Cặp nào?

**Bài 2 (đoán output):** `-7 % 2`, `-7 // 2`, `0 in set()`, `Counter([1,1])[1]`.

**Bài 3 (tự viết):** `exercise_check(arr)` bằng set một lượt.

**Bài 4 (tự viết):** `exercise_find_pair(arr)` — trả về **một cặp chỉ số** `(i, j)` với `arr[i] == 2 * arr[j]`, `i != j`, hoặc `None`. Gợi ý: dùng `dict` giá trị → chỉ số.

**Bài 5 (biến thể):** `exercise_count_pairs(arr)` — đếm số cặp chỉ số `(i, j)`, `i != j`, thoả `arr[i] == 2 * arr[j]`.

**Bài 6 (mini challenge):** `exercise_can_reorder_doubled(arr)` — LeetCode 954: mảng độ dài chẵn có thể sắp xếp lại thành các cặp `(x, 2x)` không?

In [19]:
def exercise_check(arr: list[int]) -> bool:
    # TODO
    pass

def exercise_find_pair(arr: list[int]):
    # TODO: trả (i, j) hoặc None
    pass

# assert exercise_check([10, 2, 5, 3]) and not exercise_check([0, 1])
# i, j = exercise_find_pair([10, 2, 5, 3]); assert i != j and [10, 2, 5, 3][i] == 2 * [10, 2, 5, 3][j]
# assert exercise_find_pair([3, 1, 7, 11]) is None

In [20]:
def exercise_count_pairs(arr: list[int]) -> int:
    # TODO
    pass

def exercise_can_reorder_doubled(arr: list[int]) -> bool:
    # TODO
    pass

# assert exercise_count_pairs([1, 2, 2, 4]) == 4
# assert exercise_can_reorder_doubled([4, -2, 2, -4]) is True
# assert exercise_can_reorder_doubled([3, 1, 3, 6]) is False

## 15. Gợi ý 3 mức

**Bài 1** — (1) Tìm số nào gấp đôi số nào. (2) Số âm: `-20 = 2·(-10)`. (3) Có cặp → `True`.

**Bài 4** — (1) Lưu `pos[value] = index` cho các số đã gặp. (2) Kiểm `2x` và `x/2` (khi chẵn) trong `pos`. (3) Nhớ thứ tự `(i, j)`: `arr[i]` là số **gấp đôi**.

**Bài 5** — (1) Brute force `O(n²)` là đáp án tham chiếu. (2) Với `Counter`: mỗi giá trị `v` ghép với `2v`. (3) Riêng `v = 0`: số cặp có thứ tự là `c·(c−1)`.

**Bài 6** — (1) Sắp xếp các giá trị theo **trị tuyệt đối** tăng dần. (2) Số có trị tuyệt đối nhỏ nhất bắt buộc phải ghép với gấp đôi của nó. (3) Dùng `Counter`, trừ dần `c[x]` và `c[2x]`; nếu thiếu → `False`.

## 16. Đáp án tham khảo

**Bài 1:** `True` (`-20 = 2 · (-10)`; cũng có thể kiểm thêm: `-8`, `12`, `15` không tạo cặp).

**Bài 2:** `1`, `-4`, `False`, `2`.

**Bài 5 (`[1, 2, 2, 4]`):** các cặp `(i, j)` với `arr[i] = 2·arr[j]`: `2 = 2·1` (hai số 2 → 2 cặp), `4 = 2·2` (hai số 2 → 2 cặp) ⇒ **4** cặp.

In [21]:
def answer_check(arr):
    seen = set()
    for x in arr:
        if 2 * x in seen or (x % 2 == 0 and x // 2 in seen):
            return True
        seen.add(x)
    return False

def answer_find_pair(arr):
    pos = {}
    for t, x in enumerate(arr):
        if 2 * x in pos:
            return (pos[2 * x], t)          # arr[pos] = 2 * arr[t]
        if x % 2 == 0 and x // 2 in pos:
            return (t, pos[x // 2])         # arr[t] = 2 * arr[pos]
        pos.setdefault(x, t)                # giữ chỉ số xuất hiện đầu tiên
    return None

def answer_count_pairs(arr):
    c = Counter(arr)
    total = c[0] * (c[0] - 1)
    for v, cnt in c.items():
        if v != 0:
            total += cnt * c[2 * v]
    return total

def answer_can_reorder_doubled(arr):
    c = Counter(arr)
    for x in sorted(c, key=abs):
        if c[x] == 0:
            continue
        if x == 0:
            if c[0] % 2:
                return False
            c[0] = 0
            continue
        if c[2 * x] < c[x]:
            return False
        c[2 * x] -= c[x]
        c[x] = 0
    return True

assert answer_check([-10, 12, -20, -8, 15]) is True
assert (-7 % 2, -7 // 2, 0 in set(), Counter([1, 1])[1]) == (1, -4, False, 2)
assert answer_check([10, 2, 5, 3]) and not answer_check([0, 1])
for _ in range(2000):
    arr = [rng.randint(-6, 6) for _ in range(rng.randint(2, 7))]
    p = answer_find_pair(arr)
    if check_brute(arr):
        i, j = p
        assert i != j and arr[i] == 2 * arr[j]
    else:
        assert p is None
    brute_count = sum(1 for i in range(len(arr)) for j in range(len(arr)) if i != j and arr[i] == 2 * arr[j])
    assert answer_count_pairs(arr) == brute_count
assert answer_count_pairs([1, 2, 2, 4]) == 4
assert answer_can_reorder_doubled([4, -2, 2, -4]) is True
assert answer_can_reorder_doubled([3, 1, 3, 6]) is False
assert answer_can_reorder_doubled([0, 0, 1, 2]) is True and answer_can_reorder_doubled([0, 1]) is False
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def checkIfExist(self, arr: list[int]) -> bool:
        seen = set()
        for x in arr:
            if 2 * x in seen or (x % 2 == 0 and x // 2 in seen):   # kiểm CẢ HAI chiều
                return True
            seen.add(x)                                            # thêm SAU khi kiểm
        return False
```

**Checklist:**
- [ ] Kiểm tra **trước** khi `add` (tránh ghép phần tử với chính nó).
- [ ] Kiểm cả `2x` và `x/2`; với `x/2` phải có `x % 2 == 0`.
- [ ] Đã test `[0, 0]` (True) và `[0, 1]` (False).
- [ ] Đã test số âm `[-2, -4]` và số lẻ `[7, 3]`.